In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [26]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [27]:
## PreProcessing the data

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis = 1)

In [28]:
## Encode the Gender to 0s and 1s

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [29]:
## One hot encoding the Geography class

onehot_encoder_geo = OneHotEncoder(handle_unknown = 'ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [30]:
## Now doing the concetenation

data = pd.concat([data.drop('Geography', axis = 1), geo_encoded_df], axis = 1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [31]:
## Split the data into independent features and dependent features

X = data.drop('EstimatedSalary', axis = 1)
y = data['EstimatedSalary']

In [32]:
## Split the data into testing and training sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
## Scale these features

scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [34]:
## Save the encoders and scalars

with open('onehot_encoder_geo.pkl', 'rb') as file:
  label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
  label_encoder_gender=pickle.load(file)

with open('scaler.pkl', 'rb') as file:
  scaler=pickle.load(file)

#### ANN with regression problem statement


In [35]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [36]:
## Build our ANN model

model=Sequential([
  Input(shape=(X_train.shape[1],)), ## Input layer
  Dense(64, activation='relu'), ## HL1 connected with the input layer
  Dense(32, activation='relu'), ## HL2
  Dense(1) ## Output layer, no activation funcition as it must use default linear activation funciton for the regression
])

In [37]:
## Compiling the model

model.compile(optimizer='adam', loss = 'mean_absolute_error', metrics=['mae'])

In [38]:
## Summary of the model

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [39]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

## Setup tensorboard

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callbacks = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [40]:
## Setup Early Stopping

early_stopping_callback=EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [41]:
## Train the actual model

history=model.fit(
  X_train, y_train, validation_data=(X_test, y_test ), epochs=100,
  callbacks=[early_stopping_callback, tensorboard_callbacks]
)

Epoch 1/100


I0000 00:00:1784032932.127173    3507 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_232180__.8


250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 100378.3438 - mae: 100378.3438 - val_loss: 98518.6094 - val_mae: 98518.6094
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 99620.8281 - mae: 99620.8281 - val_loss: 96970.6953 - val_mae: 96970.6953
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 96898.8125 - mae: 96898.8125 - val_loss: 92957.2188 - val_mae: 92957.2188
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 91539.3828 - mae: 91539.3828 - val_loss: 86313.6016 - val_mae: 86313.6016
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 83785.7656 - mae: 83785.7656 - val_loss: 77784.6328 - val_mae: 77784.6328
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 74802.6562 - mae: 74802.6562 - val_loss: 69096.7656 - val_mae: 69096.7656
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 66177.1172 - mae: 66177.1172 - val_loss: 61685.0703 - val_mae: 61685.0703
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 5936

In [42]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [43]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 14833), started 0:02:18 ago. (Use '!kill 14833' to kill it.)

In [44]:
## Evaluate model on the test data

test_loss, test_mae = model.evaluate(X_test, y_test)
print(f'Test MAE : {test_mae}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 50250.8281 - mae: 50250.8281
Test MAE : 50250.828125


In [45]:

model.save('regression_model.h5')